In [3]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import ZenodoSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

# WAW-TACE

this dataset sucks and i am not going to index it.


In [4]:
DATASET_NAME = "WAW-TACE"

indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])



15:53:22 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:65


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Abdomen Atlas 3',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung'
]

In [5]:
description = """
Hepatocellular carcinoma (HCC) is the most prevalent primary liver tumor and requires a multidisciplinary management approach.

Transarterial chemoembolization (TACE) emerges as a first-line therapeutic option for patients ineligible for surgical treatment.

Managing patients and predicting outcomes remains challenging, necessitating the development of novel prediction models.

Artificial intelligence (AI) shows promise as a prognostic tool, but deploying models demands a substantial amount of training data. Significant investments are directed towards providing more datasets specifically tailored for AI research.

Here, we provide the WAW-TACE dataset, which features comprehensive data from N=233 treatment-naive HCC patients treated with TACE.

This includes pre-TACE multiphase CT images annotated with N=377 hand-crafted HCC segmentations and unsupervised multi-organ masks.

Furthermore, the collection includes correspondingly extracted radiomics features and extensive clinical data, including critical outcome measures (such as progression-free survival, overall survival, and TACE technical success).

"""

In [6]:
source = ZenodoSource(
    file_type="nifti",
    source="zenodo",
    record_id="12741586",
    description=description,
    #post_download=["unzip"]
)

In [7]:
dataset_index_path = (indexed_datasets.path / DATASET_NAME)
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [8]:

print(indexed_datasets.source_config(DATASET_NAME))

ZenodoSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    post_download=['unzip'],
    source='zenodo',
    record_id='12741586',
    filenames=None,
    description='\nHepatocellular carcinoma (HCC) is the most prevalent primary liver tumor and requires a 
multidisciplinary management approach.\n\nTransarterial chemoembolization (TACE) emerges as a first-line 
therapeutic option for patients ineligible for surgical treatment.\n\nManaging patients and predicting outcomes 
remains challenging, necessitating the development of novel prediction models.\n\nArtificial intelligence (AI) 
shows promise as a prognostic tool, but deploying models demands a substantial amount of training data. Significant
investments are directed towards providing more datasets specifically tailored for AI research.\n\nHere, we provide
the WAW-TACE dataset, which features comprehensive data from N=233 treatment-naive HCC patients treated with 
TACE.\n\nThis includes pre-TACE multiphase CT images annotated with N=377 hand-crafted HCC segmentations and 
unsupervised multi-organ masks.\n\nFurthermore, the collection includes correspondingly extracted radiomics 
features and extensive clinical data, including critical outcome measures (such as progression-free survival, 
overall survival, and TACE technical success).\n\n'
)

In [9]:
temp_data_path = Path("temp_data") / DATASET_NAME
temp_data_path.mkdir(parents=True, exist_ok=True)
#downloader = indexed_datasets.downloader(DATASET_NAME).download(temp_data_path)

In [12]:
import zipfile
from glob import glob
import os

def extract_zip(zip_path, extract_to=None):
    """
    Extract a zip file to a specified directory
    
    Args:
        zip_path: Path to the zip file
        extract_to: Directory to extract to (default: same directory as zip)
    """
    if extract_to is None:
        extract_to = os.path.dirname(zip_path)
    
    # Create extraction directory if it doesn't exist
    os.makedirs(extract_to, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        print(f"Extracted {zip_path} to {extract_to}")

for zip_file in glob(f"{temp_data_path}/*.zip"):
    print(zip_file)
    # Get the base name without extension to create a folder with the same name
    zip_name = os.path.splitext(os.path.basename(zip_file))[0]
    extract_to = os.path.join(temp_data_path, zip_name)
    extract_zip(zip_file, extract_to)

temp_data/WAW-TACE/ct_scans_3_4_wawtace_09_05_24.zip

Extracted temp_data/WAW-TACE/ct_scans_3_4_wawtace_09_05_24.zip to temp_data/WAW-TACE/ct_scans_3_4_wawtace_09_05_24

temp_data/WAW-TACE/ct_scans_2_4_wawtace_09_05_24.zip

Extracted temp_data/WAW-TACE/ct_scans_2_4_wawtace_09_05_24.zip to temp_data/WAW-TACE/ct_scans_2_4_wawtace_09_05_24

temp_data/WAW-TACE/organ_masks_wawtace_09_05_2024.zip

Extracted temp_data/WAW-TACE/organ_masks_wawtace_09_05_2024.zip to 
temp_data/WAW-TACE/organ_masks_wawtace_09_05_2024

... (2 outputs left out)

temp_data/WAW-TACE/ct_scans_4_4_wawtace_09_05_24.zip

Extracted temp_data/WAW-TACE/ct_scans_4_4_wawtace_09_05_24.zip to temp_data/WAW-TACE/ct_scans_4_4_wawtace_09_05_24

temp_data/WAW-TACE/tumor_masks_wawtace_v1_08_05_2024.zip

Extracted temp_data/WAW-TACE/tumor_masks_wawtace_v1_08_05_2024.zip to 
temp_data/WAW-TACE/tumor_masks_wawtace_v1_08_05_2024

In [13]:
crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="ct_scans{drop1}/{drop2}/{patient_id}_{phase}_scan.nii.gz",
    mask_name_pattern="{ROINames}_masks{drop2}/{drop3}/{patient_id}_{phase}_total_segmentator.nii.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=False,
)
crawler.crawl()

index_df = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")

crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="ct_scans{drop1}/{drop2}/{patient_id}_{phase}_scan.nii.gz",
    mask_name_pattern="{ROINames}_masks{drop2}/{drop3}/{drop4}/{patient_id}_{phase}_{tumour_number}_tumor_seg.nnrd",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=False,
)
crawler.crawl()

index_df2 = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")

# drop all scans from the second crawl
index_df2 = index_df2[index_df2["file_type"] != "scan"]


12:46:35 [warning  ] Some files did not match any pattern. [imgtools] unmatched_count=98 total_count=1708 example_paths=['organ_masks_wawtace_09_05_2024/303/303_0_total_segmentator.3_WATROBA_3_FAZY_20180524182449_2.nii.nii.gz', 'organ_masks_wawtace_09_05_2024/304/304_0_total_segmentator.2_Jb_M_2,5_2.5mm_2_fazy_SmartPrep_20180706144617_2.nii.nii.gz', 'organ_masks_wawtace_09_05_2024/304/304_2_total_segmentator.2_Jb_M_2,5_2.5mm_2_fazy_SmartPrep_20180706144617_4.nii.nii.gz', 'organ_masks_wawtace_09_05_2024/304/304_3_total_segmentator.2_Jb_M_2,5_2.5mm_2_fazy_SmartPrep_20180706144617_5.nii.nii.gz', 'organ_masks_wawtace_09_05_2024/310/310_0_total_segmentator.3_WATROBA_3_FAZY_20180723090356_2.nii.nii.gz'] patterns_tried=['ct_scans{drop1}/{drop2}/{patient_id}_{phase}_scan.nii.gz', '{ROINames}_masks{drop2}/{drop3}/{patient_id}_{phase}_total_segmentator.nii.nii.gz'] hint='Check that patterns include all path segments between root and filename.' call=parse_niftis._log_unmatched_summary:189
12:46:3

In [ ]:
index_df_combined = pd.concat([index_df, index_df2], axis=0, ignore_index=True)
print(len(index_df_combined))